# Scraping News Sources: Heute, ORF, MeinBezirk

### This script contains all the syntax for scraping:
- https://www.heute.at/artikelsuche
- https://orfarchiv.news/
- [Wikipedia AT district names](https://en.wikipedia.org/wiki/Districts_of_Austria)
- https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Austria
- [meinbezirk.at](https://www.meinbezirk.at/s/search)

In [ ]:
#!pip install playwright
#!pip install playwright-stealth

## 1. Heute

In [ ]:
# script to extract all the links from the heute.at search results for "hochwasser 2024"
# running this kernel opens up the browser window! (heute has really strong bot protection, so some things I did manually)

import asyncio
from playwright.async_api import async_playwright

async def surgical_heute_scraper():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        print("--- ACTION REQUIRED ---")
        print("1. Search manually for 'hochwasser 2024'.")
        print("2. Scroll and click 'Mehr Ergebnisse laden'.")
        
        await page.goto("https://www.heute.at/artikelsuche")
        
        all_links = set()
        
        while True:
            await page.wait_for_timeout(3000)
            
            # This targets only links INSIDE the result grid
            # based on the class 'article' you found earlier.
            results = await page.evaluate("""
                () => {
                    const container = document.querySelector('.search_results_container');
                    if (!container) return [];
                    const items = Array.from(container.querySelectorAll('a.article'));
                    return items.map(a => a.href);
                }
            """)
            
            new_found = 0
            for url in results:
                if url not in all_links:
                    all_links.add(url)
                    new_found += 1
            
            if new_found > 0:
                print(f"✅ Found {new_found} new articles. Total unique: {len(all_links)}")
                # Auto-save current progress
                with open("heute_flood_results.txt", "w") as f:
                    for l in sorted(all_links):
                        f.write(l + "\n")
            elif len(results) > 0:
                print(f"Watching {len(results)} articles on screen. Click 'Mehr laden' to see more.")
            else:
                print("Waiting for you to hit 'Suchen'...")

await surgical_heute_scraper()

--- ACTION REQUIRED ---
1. Search manually for 'hochwasser 2024'.
2. Scroll and click 'Mehr Ergebnisse laden'.
Waiting for you to hit 'Suchen'...
Waiting for you to hit 'Suchen'...
Waiting for you to hit 'Suchen'...
Waiting for you to hit 'Suchen'...
Waiting for you to hit 'Suchen'...
✅ Found 50 new articles. Total unique: 50
Watching 50 articles on screen. Click 'Mehr laden' to see more.
✅ Found 50 new articles. Total unique: 100
Watching 100 articles on screen. Click 'Mehr laden' to see more.
✅ Found 50 new articles. Total unique: 150
Watching 150 articles on screen. Click 'Mehr laden' to see more.
Watching 150 articles on screen. Click 'Mehr laden' to see more.
✅ Found 50 new articles. Total unique: 200
Watching 200 articles on screen. Click 'Mehr laden' to see more.
✅ Found 50 new articles. Total unique: 250
Watching 250 articles on screen. Click 'Mehr laden' to see more.
✅ Found 50 new articles. Total unique: 300
✅ Found 50 new articles. Total unique: 350
Watching 350 articles on 

CancelledError: 

In [20]:
# checking if the links were saved correctly
f = open("/home/ina/Desktop/thesis/Thesis-Project/article-scraping/output/heute_flood_results.txt", "r")
print(f.read(150))

https://www.heute.at/s/1-mio-euro-schaden-unternehmer-wartet-auf-flut-hilfe-120076920
https://www.heute.at/s/10-tage-vor-wahl-platzt-elefantenrunde-au


In [ ]:
# confirm the number of articles found
with open("/home/ina/Desktop/thesis/Thesis-Project/article-scraping/output/heute_flood_results.txt", 'r') as fp:
    lines = sum(1 for line in fp)
    print('Total Number of lines:', lines)

Total Number of lines: 800


In [ ]:
# save each link to a list as list item
with open("/home/ina/Desktop/heute_flood_results.txt", 'r') as input_file:
    articles = []
    for line in input_file:
        articles.append(line.strip())

In [ ]:
# scrape each article and save results to a DataFrame
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Initialize an empty list (more efficient than pd.concat in a loop)
results = []

for i in articles:
    url = i.strip()
    headers = {"User-Agent": "Mozilla/5.0"}
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Check if the request was successful
        soup = BeautifulSoup(response.text, "html.parser")

        # Use helper variables to avoid AttributeError
        h1_tag = soup.find("h1", class_="headline")
        sub_tag = soup.find("div", class_="sub-headline")
        date_tag = soup.find("div", class_="author-time")
        
        # Extract text safely with a fallback value
        headline = h1_tag.get_text(strip=True) if h1_tag else "N/A"
        sub_headline = sub_tag.get_text(strip=True) if sub_tag else "N/A"
        date = date_tag.get_text(strip=True) if date_tag else "N/A"

        # Body text (find_all returns an empty list if nothing found, so this is safe)
        p_tags = soup.find_all("p", class_="gb-p")
        body_text = " ".join([p.get_text(strip=True) for p in p_tags]) if p_tags else "N/A"

        # Append dictionary to list
        results.append({
            "title": headline,
            "sub_header": sub_headline,
            "body": body_text,
            "date": date,
            "url": url
        })
        
        # Be nice: wait 1 second between requests so you don't get banned
        time.sleep(1)

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        continue

# Create the DataFrame once at the end
df = pd.DataFrame(results)

In [ ]:
# save articles to csv
# import csv
# with open('heute_flood_articles.csv', 'w', newline='') as csvfile:
#     fieldnames = ['title', 'sub_header', 'body', 'date', "url"]
#     writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(df.to_dict('records'))

## 2. Districts from Wikipedia

<a href="/wiki/G%C3%BCssing_(district)" class="mw-redirect" title="Güssing (district)">Güssing</a>

In [90]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Districts_of_Austria"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# 1. Find the specific table (it's a 'wikitable sortable')
table = soup.find("table", class_="wikitable")

districts = []

# 2. Iterate through rows, skipping the header (the first row)
for row in table.find_all("tr")[1:]:
    cells = row.find_all(["td", "a"])
    
    # Ensure the row has enough columns before accessing index 2
    if len(cells) > 2:
        # Get text from the 3rd column (index 2)
        name = cells[2].get_text(strip=True)
        districts.append(name)

# Print first 5 to verify
print(districts[:5])

['101', 'Rust', 'Eisenstadt-Umgebung', 'Güssing', 'Jennersdorf']


In [ ]:
# Some of the larger towns didn't scrape correctly (were saved as a number instead), so I added them manually
districts = districts + ["Eisenstadt", "Klagenfurt am Wörthersee", "Linz", "Salzburg", "Graz", "Innsbruck", "Wien"]

# remove all items that are numbers (these are the ones that didn't scrape correctly)
districts = list(filter(lambda x: "0" not in x, districts))
len(districts)

In [ ]:
# save districts to a txt file
with open("districts.txt", "w") as f:
    for l in districts:
        f.write(l + "\n")

## 3. Towns from Wikipedia

In [21]:
import requests
from bs4 import BeautifulSoup
import re

headers = {"User-Agent": "Mozilla/5.0"}
url = "https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Austria"
response = requests.get(url, headers=headers )
soup = BeautifulSoup(response.text, "html.parser")

# Find all tables with the 'wikitable' class
all_wikitables = soup.find_all("table", class_="wikitable")

town_names = []

# Skip the first table (the summary table)
for table in all_wikitables[1:10]:
    for row in table.find_all("tr")[1:]: # Skip headers
        cells = row.find_all("a")
        if cells:
            # Usually the name is in the first column (index 0)
            name_text = cells[0].get_text(strip=True)
            
            # CLEANING: Remove citations like [1] or symbols like *
            clean_name = re.sub(r'\[.*?\]|\*', '', name_text)
            town_names.append(clean_name)

print(town_names)

['Andau', 'Antau', 'Apetlon', 'Bad Sauerbrunn', 'Bad Tatzmannsdorf', 'Badersdorf', 'Baumgarten', 'Bernstein', 'Bildein', 'Bocksdorf', 'Breitenbrunn am Neusiedler See', 'Bruckneudorf', 'Burgauberg-Neudauberg', 'Deutsch Jahrndorf', 'Deutsch Kaltenbrunn', 'Deutsch Schützen-Eisenberg', 'Deutschkreutz', 'Donnerskirchen', 'Draßburg', 'Draßmarkt', 'Eberau', 'Edelstal', 'Eisenstadt', 'Eltendorf', 'Forchtenstein', 'Frankenau-Unterpullendorf', 'Frauenkirchen', 'Gattendorf', 'Gerersdorf-Sulz', 'Gols', 'Grafenschachen', 'Großhöflein', 'Großmürbisch', 'Großpetersdorf', 'Großwarasdorf', 'Güssing', 'Güttenbach', 'Hackerberg', 'Halbturn', 'Hannersdorf', 'Heiligenbrunn', 'Heiligenkreuz im Lafnitztal', 'Heugraben', 'Hirm', 'Horitschon', 'Hornstein', 'Illmitz', 'Inzenhof', 'Jabing', 'Jennersdorf', 'Jois', 'Kaisersdorf', 'Kemeten', 'Kittsee', 'Kleinmürbisch', 'Klingenbach', 'Kobersdorf', 'Kohfidisch', 'Königsdorf', 'Krensdorf', 'Kukmirn', 'Lackenbach', 'Lackendorf', 'Leithaprodersdorf', 'Litzelsdorf', 'Lo

In [ ]:
# save towns to a txt file
with open("towns.txt", "w") as f:
    for l in town_names:
        f.write(l + "\n")

## 4. ORF

ORF doesn't store its articles as a link, but they are saved directly on the archive website. This is why I directly scrape them from there.

In [ ]:
import asyncio
import json
from playwright.async_api import async_playwright

# Persists data even if the script stops
final_results = []

async def scrape_orf_archiv_automated():
    global final_results
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        await page.goto("https://orfarchiv.news/")

        print("--- PREPARATION PHASE ---")
        print("You have 30 seconds to:")
        print("1. Type 'hochwasser' and search.")
        print("2. Set your date filters.")
        print("3. Clear any overlays or cookie banners.")
        
        # Countdown timer
        for i in range(30, 0, -1):
            if i % 5 == 0 or i <= 5:
                print(f"Starting in {i} seconds...")
            await asyncio.sleep(1)
        
        print("\n--- SCRAPING PHASE STARTING ---")
        
        seen_bodies = set()

        try:
            while True:
                if page.is_closed():
                    break

                items = await page.query_selector_all("li.flex-col")

                for item in items:
                    # check if already expanded
                    body_container = await item.query_selector("#ss-storyContent")
                    
                    if not body_container:
                        header_button = await item.query_selector("header[role='button']")
                        if header_button:
                            try:
                                # scroll_into_view_if_needed helps prevent clicking off-screen elements
                                await header_button.scroll_into_view_if_needed()
                                await header_button.click()
                                await asyncio.sleep(0.7) # Wait for animation
                                body_container = await item.query_selector("#ss-storyContent")
                            except Exception:
                                continue # Skip if click fails (e.g. element moved during scroll)

                    if body_container:
                        body_paragraphs = await body_container.query_selector_all("p")
                        body_text = "\n".join([await p.inner_text() for p in body_paragraphs])

                        if body_text and body_text not in seen_bodies:
                            title_el = await item.query_selector("h3")
                            title = await title_el.inner_text() if title_el else "Unknown"

                            meta_el = await item.query_selector("span.text-gray-600")
                            metadata = await meta_el.inner_text() if meta_el else ""

                            abstract_el = await item.query_selector("strong")
                            abstract = await abstract_el.inner_text() if abstract_el else ""

                            caption_elements = await item.query_selector_all("figcaption.caption")
                            captions_list = [await c.inner_text() for c in caption_elements]
                            all_captions = "; ".join(captions_list)

                            entry = {
                                "title": title.strip(),
                                "metadata": metadata.replace("\n", " ").strip(),
                                "abstract": abstract.strip(),
                                "figcaption": all_captions.strip(),
                                "body": body_text.strip()
                            }

                            final_results.append(entry)
                            seen_bodies.add(body_text)
                            print(f"✅ Captured: {title[:50]}...")

                            with open("orf_scraped_data.json", "w", encoding="utf-8") as f:
                                json.dump(final_results, f, ensure_ascii=False, indent=4)

                # Slowed down slightly to be gentler on the CPU/Browser
                await asyncio.sleep(2)
                
        except Exception as e:
            print(f"\nInteraction paused: {e}")
        finally:
            if not page.is_closed():
                await browser.close()

# Run the scraper
await scrape_orf_archiv_automated()

print(f"\nSession finished. Total articles: {len(final_results)}")

# total articles: 281; I'll still have to filter them for date and relevance

In [ ]:
# # save as csv

# import csv

# with open('orf_flood_articles.csv', 'w', newline='') as csvfile:
#     fieldnames = ['title', "metadata", "abstract", "figcaption", 'body']
#     writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(final_results)

## 5. MeinBezirk

In [ ]:
#  capturing the urls first
# this kernel also saves it to a txt file

import asyncio
import json
from playwright.async_api import async_playwright

# This will store our found URLs
meinbezirk_urls = set()

async def scrape_meinbezirk_links():
    global meinbezirk_urls
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        # 1. Navigate to the search page
        await page.goto("https://www.meinbezirk.at/s/search")

        print("--- PREPARATION PHASE ---")
        print("1. Type 'hochwasser 2024' into the search box.")
        print("2. Press Enter/Search.")
        
        # 30-second buffer for your manual input
        for i in range(30, 0, -1):
            if i % 5 == 0 or i <= 5:
                print(f"Starting link capture in {i} seconds...")
            await asyncio.sleep(1)

        print("\n--- CAPTURING LINKS ---")
        print("Scroll down or click next pages; I will grab all unique full URLs I see.")

        try:
            while True:
                if page.is_closed():
                    break

                # Target the actual links in the Google Custom Search results
                # These are usually inside 'a.gs-title'
                links = await page.query_selector_all("a.gs-title")
                
                new_found_in_loop = 0
                for link in links:
                    url = await link.get_attribute("href")
                    
                    # Basic filter to ensure it's an actual article link and not an internal Google redirect
                    if url and "meinbezirk.at" in url and url not in meinbezirk_urls:
                        # Sometimes Google CSE URLs have tracking parameters, we can clean them if needed
                        clean_url = url.split("?")[0].split("#")[0]
                        
                        if clean_url not in meinbezirk_urls:
                            meinbezirk_urls.add(clean_url)
                            new_found_in_loop += 1
                            print(f"🔗 Found: {clean_url}")

                if new_found_in_loop > 0:
                    # Save progress to a text file
                    with open("meinbezirk_links.txt", "w", encoding="utf-8") as f:
                        for l in sorted(meinbezirk_urls):
                            f.write(l + "\n")
                
                # Check for new results every 2 seconds
                await asyncio.sleep(2)

        except Exception as e:
            print(f"Session ended: {e}")
        finally:
            if not page.is_closed():
                await browser.close()

# Run the link scraper
await scrape_meinbezirk_links()

print(f"\nFinished! Total unique URLs captured: {len(meinbezirk_urls)}")

In [27]:
cd /home/ina/Desktop/thesis/Thesis-Project/article-scraping/output

/home/ina/Desktop/thesis/Thesis-Project/article-scraping/output


In [39]:
meinbezirk_urls = []
with open("meinbezirk_links.txt", 'r') as file:
    for line in file:
        meinbezirk_urls.append(line.strip())
        print(line.strip())

https://www.meinbezirk.at/baden/c-lokales/grosse-hilfsbereitschaft-fuer-vereine-beim-hochwasser-2024_a7540298
https://www.meinbezirk.at/baden/c-lokales/hochwasser-an-der-schwechat-piesting-und-triesting_a6898162
https://www.meinbezirk.at/bruck-an-der-leitha/c-lokales/bilanz-nach-ueberflutungen-im-bezirk-bruck_a6912462
https://www.meinbezirk.at/c-lokales/42-millionen-euro-fuer-oesterreich-nach-hochwasser-2024_a7874288
https://www.meinbezirk.at/enns/c-lokales/aktuelle-hochwasser-situation-in-der-region-enns_a6897121
https://www.meinbezirk.at/gmuend/c-gedanken/nach-der-flut-ist-vor-der-neuen-flut_a7569539
https://www.meinbezirk.at/gmuend/c-lokales/erste-hochwasser-bilanz-im-bezirk-gmuend_a6900337
https://www.meinbezirk.at/gmuend/c-lokales/hochwasser-2024-im-fokus-bei-teichwirterunde-spezial_a7743746
https://www.meinbezirk.at/horn/c-lokales/entspannung-der-lage-informationsschreiben_a6901200
https://www.meinbezirk.at/kirchdorf/c-regionauten-community/rueckhaltebecken-krems-au-hat-sich-112-

In [40]:
meinbezirk_urls

['https://www.meinbezirk.at/baden/c-lokales/grosse-hilfsbereitschaft-fuer-vereine-beim-hochwasser-2024_a7540298',
 'https://www.meinbezirk.at/baden/c-lokales/hochwasser-an-der-schwechat-piesting-und-triesting_a6898162',
 'https://www.meinbezirk.at/bruck-an-der-leitha/c-lokales/bilanz-nach-ueberflutungen-im-bezirk-bruck_a6912462',
 'https://www.meinbezirk.at/c-lokales/42-millionen-euro-fuer-oesterreich-nach-hochwasser-2024_a7874288',
 'https://www.meinbezirk.at/enns/c-lokales/aktuelle-hochwasser-situation-in-der-region-enns_a6897121',
 'https://www.meinbezirk.at/gmuend/c-gedanken/nach-der-flut-ist-vor-der-neuen-flut_a7569539',
 'https://www.meinbezirk.at/gmuend/c-lokales/erste-hochwasser-bilanz-im-bezirk-gmuend_a6900337',
 'https://www.meinbezirk.at/gmuend/c-lokales/hochwasser-2024-im-fokus-bei-teichwirterunde-spezial_a7743746',
 'https://www.meinbezirk.at/horn/c-lokales/entspannung-der-lage-informationsschreiben_a6901200',
 'https://www.meinbezirk.at/kirchdorf/c-regionauten-community/r

In [ ]:
# scraping all the website infos and save to dataframe

mein_bezirk_results = []
headers = {"User-Agent": "Mozilla/5.0"}


for link in meinbezirk_urls:
    #response = requests.get(f"{link}", headers=headers )
    response = requests.get(link, headers=headers )
    soup = BeautifulSoup(response.text, "html.parser")
    
    
    # title and url
    title = soup.title.string
    
    # date
    meta_ul = soup.find("ul", class_="content-meta")
    
    if not meta_ul:
        print(f"⚠️ No metadata found for {link}")
        date = "N/A"
    else:
        date_text = meta_ul.find("li").get_text(strip=True)
        # To get just the date without the time:
        date = date_text.split(",")[0]
    
    # body text
    all_ps = soup.find("div", id="content-main").find_all("p")
    cleaned_text_list = []

    for p in all_ps:
        # Prüfen, ob ein <b> Tag im Paragraph existiert
        if p.find("b"):
            continue  # Springe zum nächsten p, wenn Fettdruck gefunden wurde
        
        # Text bereinigen und hinzufügen
        text_p = p.get_text(strip=True)
        if text_p: # Verhindert leere Zeilen
            cleaned_text_list.append(text_p)

    text = "\n\n".join(cleaned_text_list)
    
    mein_bezirk_results.append({
        "title": title,
        "url": link,
        "date": date,
        "text": text
    })

⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-gm%C3%BCnd
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6-amstetten
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6-hollabrunn
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6/11
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6/15
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6/16
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6/18
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6/2
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6/20
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-n%C3%B6/5
⚠️ No metadata found for https://www.meinbezirk.at/tag/hochwasser-2024-neunkirchen
⚠️ No metadat

In [43]:
len(mein_bezirk_results)
len(meinbezirk_urls)

100

In [ ]:
# # save as csv

# import csv

# with open('bezirk_flood_articles.csv', 'w', newline='') as csvfile:
#     fieldnames = ['title', "url", "date", "text"]
#     writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(mein_bezirk_results)